In [ ]:
"""You have 10 million transaction rows. You want to calculate total revenue per merchant. But 10 merchants — M00001 through M00010 — account for 
   70% of all 10 million transactions. The remaining 490 merchants share only 30%.

When Spark processes this groupBy on merchant_id, it assigns each merchant's data to a partition. The partitions holding the 10 hot merchants have 
700,000 rows each on average. Every other partition has roughly 600 rows.
"""
#==>Before — Without Salting
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum

# ── Create Spark Session ─────────────────────────────
spark = SparkSession.builder \
    .appName("Skew Handling with Salting") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# ── Read Data ───────────────────────────────────────
df = spark.read.option("mergeSchema", "true").parquet(
    r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions"
)
# ── raw total_revenue with skew data without salting───────────────────────────────────────
df.groupBy("merchant_id").agg(sum("amount").alias('total_revenue')).show()

+-----------+--------------------+
|merchant_id|       total_revenue|
+-----------+--------------------+
|     M00354|2.6818512279999994E7|
|     M00080|2.6041952930000003E7|
|     M00043|2.5272492569999997E7|
|     M00413|2.7058869099999998E7|
|     M00110| 2.589143937000001E7|
|     M00288|2.6076336549999993E7|
|     M00012|2.5935203819999993E7|
|     M00345|2.7028263189999994E7|
|     M00374|       2.627767088E7|
|     M00067|2.6141118169999994E7|
|     M00033|2.5853458070000004E7|
|     M00403|2.6937652149999995E7|
|     M00344|       2.616272115E7|
|     M00273|       2.807995548E7|
|     M00161|        2.77921618E7|
|     M00453|       2.600125563E7|
|     M00096|2.8026661819999997E7|
|     M00465| 2.628371847000001E7|
|     M00278|2.5756272850000005E7|
|     M00499|       2.535696197E7|
+-----------+--------------------+
only showing top 20 rows



In [ ]:
#==>After — With Salting
from pyspark.sql.functions import col, count, rand, concat, lit, split, sum, broadcast
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Skew Handling with Salting") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "50") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df = spark.read.option("mergeSchema", "true").parquet(
    r"D:\data engineer\pyspark_practice\pyspark_scenarios\dataset1_transactions\transactions"
)
# Detect skew
skew_df = df.groupBy("merchant_id").agg(count("*").alias("cnt")).cache()

hot_keys = skew_df.filter(col("cnt") > 100000).select("merchant_id")

# Split data
df1 = df.join(broadcast(hot_keys), "merchant_id", "left_semi")
df2 = df.join(broadcast(hot_keys), "merchant_id", "left_anti")

# Apply salting
salt_df = df1.withColumn(
    "salt_key",
    concat(col("merchant_id"), lit("_"), (rand()*10).cast("int"))
)
salt_group = salt_df.groupBy("salt_key") \
    .agg(sum("amount").alias("salt_amount"))

# De-salt
total_salted = salt_group.withColumn(
    "merchant_id",
    split(col("salt_key"), "_").getItem(0)
).groupBy("merchant_id") \
 .agg(sum("salt_amount").alias("total_amount"))

# Normal aggregation
total_without_salted = df2.groupBy("merchant_id") \
    .agg(sum("amount").alias("total_amount"))
# Final result
df_result = total_salted.unionByName(total_without_salted) \
    .groupBy("merchant_id") \
    .agg(sum("total_amount").alias('total_revenue'))

df_result.show()

+-----------+--------------------+
|merchant_id|       total_revenue|
+-----------+--------------------+
|     M00003|     3.04201465143E9|
|     M00010| 3.018745371920002E9|
|     M00008|3.0046319164199996E9|
|     M00002|3.0247393003100014E9|
|     M00001|3.0195015451700015E9|
|     M00009|3.0237288295700016E9|
|     M00006|3.0270511329199996E9|
|     M00005| 3.004247678479999E9|
|     M00004|3.0092930939899993E9|
|     M00007|2.9983132212499995E9|
|     M00481|2.5711687130000018E7|
|     M00119|2.7388665410000004E7|
|     M00159| 2.533208608000001E7|
|     M00379| 2.575955176000001E7|
|     M00354|2.6818512279999994E7|
|     M00080|2.6041952930000003E7|
|     M00178|2.7094744519999996E7|
|     M00148|       2.670694878E7|
|     M00192|2.5747871389999993E7|
|     M00043|2.5272492569999997E7|
+-----------+--------------------+
only showing top 20 rows

